# 🇬🇪 GeoTutor: Georgian PDF → Personal AI Tutor

**Theme:** Digital Equity + Future of Education  
**Tech:** Gemma 3 + Kaggle GPU + PWA Offline  
**Goal:** Transform any Georgian PDF into interactive Quiz, Flashcards & Memory Game

---

## Problem
- 95% of world languages lack AI learning tools
- 40% of rural Georgia has limited internet
- Students get PDFs but no interactive way to study them

## Solution
1. Upload any Georgian PDF
2. AI generates Quiz + Flashcards + Memory Game
3. Export as offline PWA — works without internet

## 1. Setup & Dependencies

In [ ]:
!pip install -q PyPDF2 transformers accelerate bitsandbytes sentencepiece protobuf

In [ ]:
import json
import re
import os
from pathlib import Path

import torch
from PyPDF2 import PdfReader
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from IPython.display import display, HTML, JSON

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else '')

## 2. PDF Text Extraction

Upload your Georgian PDF and extract text from it.

In [ ]:
def extract_pdf_text(pdf_path: str) -> str:
    """Extract text from PDF, handling Georgian unicode."""
    reader = PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text and text.strip():
            pages.append(text.strip())
    
    full_text = "\n\n".join(pages)
    print(f"Extracted {len(pages)} pages, {len(full_text)} characters")
    return full_text


def chunk_text(text: str, max_chars: int = 3000) -> list[str]:
    """Split text into chunks for LLM context window."""
    paragraphs = text.split("\n\n")
    chunks = []
    current = ""
    
    for para in paragraphs:
        if len(current) + len(para) > max_chars and current:
            chunks.append(current.strip())
            current = para
        else:
            current += "\n\n" + para
    
    if current.strip():
        chunks.append(current.strip())
    
    print(f"Split into {len(chunks)} chunks")
    return chunks

In [ ]:
# --- Upload your PDF here ---
# On Kaggle: Add your PDF as a dataset, then use the path below
# Example: /kaggle/input/your-dataset/your-file.pdf

PDF_PATH = "/kaggle/input/your-pdf/document.pdf"  # <-- CHANGE THIS

# For testing without a PDF, use this sample Georgian text:
SAMPLE_TEXT = """
თავი 1: Python-ის საფუძვლები

Python არის მაღალი დონის პროგრამირების ენა, რომელიც შეიქმნა 1991 წელს გვიდო ვან როსუმის მიერ.
Python-ის მთავარი უპირატესობებია: მარტივი სინტაქსი, დიდი სტანდარტული ბიბლიოთეკა და 
კროს-პლატფორმულობა.

ცვლადები Python-ში:
- int: მთელი რიცხვები (მაგ: x = 5)
- float: წილადი რიცხვები (მაგ: y = 3.14)
- str: ტექსტური სტრიქონები (მაგ: name = "გიორგი")
- bool: ლოგიკური მნიშვნელობები (True / False)

თავი 2: ციკლები და პირობები

if/elif/else პირობითი ოპერატორი საშუალებას გვაძლევს კოდის სხვადასხვა ნაწილი შევასრულოთ
პირობის მიხედვით.

for ციკლი გამოიყენება იტერაციისთვის - მაგალითად სიაში ელემენტების გადასარჩევად.
while ციკლი მეორდება სანამ პირობა ჭეშმარიტია.

თავი 3: ფუნქციები

ფუნქცია არის კოდის ბლოკი, რომელიც ასრულებს კონკრეტულ ამოცანას.
def keyword-ით ვქმნით ფუნქციას. ფუნქციას შეიძლება ჰქონდეს პარამეტრები და დააბრუნოს მნიშვნელობა.

Lambda ფუნქციები არის ანონიმური ფუნქციები ერთი გამოსახულებით:
square = lambda x: x ** 2

თავი 4: სიები და ლექსიკონები

სია (list) არის მოწესრიგებული კოლექცია: fruits = ["ვაშლი", "მსხალი", "ყურძენი"]
ლექსიკონი (dict) არის key-value წყვილების კოლექცია: student = {"სახელი": "ნინო", "ასაკი": 20}

List comprehension: squares = [x**2 for x in range(10)]
Dictionary comprehension: {k: v for k, v in items}

თავი 5: შეცდომების დამუშავება

try/except ბლოკი გამოიყენება შეცდომების დასაჭერად:
try:
    result = 10 / 0
except ZeroDivisionError:
    print("ნულზე გაყოფა შეუძლებელია")

finally ბლოკი ყოველთვის სრულდება, მიუხედავად შეცდომისა.
"""

# Try PDF first, fallback to sample
if os.path.exists(PDF_PATH):
    source_text = extract_pdf_text(PDF_PATH)
else:
    print("PDF not found, using sample Georgian text for demo")
    source_text = SAMPLE_TEXT

chunks = chunk_text(source_text)
print(f"\nFirst 500 chars:\n{source_text[:500]}")

## 3. Load Gemma Model

Loading Gemma 3 4B with 4-bit quantization — fits in Kaggle's T4 GPU (16GB VRAM).

In [ ]:
MODEL_ID = "google/gemma-3-4b-it"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
)
print("Model loaded!")

In [ ]:
def generate(prompt: str, max_new_tokens: int = 2048) -> str:
    """Generate text with Gemma."""
    messages = [
        {"role": "user", "content": prompt}
    ]
    
    input_ids = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )
    
    result = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)
    return result.strip()

# Quick test
test = generate("Say 'GeoTutor works!' in Georgian")
print(f"Test: {test}")

## 4. Quiz Generation

Generate multiple-choice questions from PDF content in Georgian.

In [ ]:
QUIZ_PROMPT = """You are a Georgian language education assistant.
Based on the following text, generate exactly {n} multiple-choice questions in Georgian.

TEXT:
{text}

Return ONLY valid JSON array. Each item must have:
- "question": the question in Georgian
- "options": array of 4 options [A, B, C, D] in Georgian
- "correct": index 0-3 of correct answer
- "explanation": brief explanation in Georgian

Example format:
[
  {{
    "question": "რა არის Python?",
    "options": ["პროგრამირების ენა", "ოპერაციული სისტემა", "ბრაუზერი", "მონაცემთა ბაზა"],
    "correct": 0,
    "explanation": "Python არის მაღალი დონის პროგრამირების ენა"
  }}
]

Generate {n} questions. Return ONLY the JSON array, nothing else."""


def generate_quiz(text: str, n: int = 5) -> list[dict]:
    """Generate quiz questions from text chunk."""
    prompt = QUIZ_PROMPT.format(text=text[:3000], n=n)
    response = generate(prompt)
    
    # Extract JSON from response
    try:
        # Try to find JSON array in response
        match = re.search(r'\[.*\]', response, re.DOTALL)
        if match:
            questions = json.loads(match.group())
            return questions
    except json.JSONDecodeError:
        pass
    
    print(f"Warning: Could not parse quiz JSON, raw response:\n{response[:500]}")
    return []

In [ ]:
# Generate quiz from each chunk
all_questions = []

for i, chunk in enumerate(chunks):
    print(f"Generating quiz for chunk {i+1}/{len(chunks)}...")
    questions = generate_quiz(chunk, n=5)
    all_questions.extend(questions)
    print(f"  → {len(questions)} questions generated")

print(f"\nTotal: {len(all_questions)} quiz questions")

# Preview
for i, q in enumerate(all_questions[:3]):
    print(f"\nQ{i+1}: {q.get('question', 'N/A')}")
    for j, opt in enumerate(q.get('options', [])):
        marker = '✅' if j == q.get('correct', -1) else '  '
        print(f"  {marker} {chr(65+j)}) {opt}")

## 5. Flashcard Generation

Generate study flashcards — term/concept on front, definition on back.

In [ ]:
FLASHCARD_PROMPT = """You are a Georgian language education assistant.
Based on the following text, generate exactly {n} flashcards in Georgian.

TEXT:
{text}

Return ONLY valid JSON array. Each item must have:
- "front": key term or concept (short, Georgian)
- "back": definition or explanation (1-2 sentences, Georgian)
- "category": topic category in Georgian

Example:
[
  {{
    "front": "ცვლადი (Variable)",
    "back": "ცვლადი არის სახელი, რომელიც მიენიჭება მეხსიერების უბანს მონაცემების შესანახად.",
    "category": "საფუძვლები"
  }}
]

Generate {n} flashcards. Return ONLY the JSON array."""


def generate_flashcards(text: str, n: int = 8) -> list[dict]:
    """Generate flashcards from text chunk."""
    prompt = FLASHCARD_PROMPT.format(text=text[:3000], n=n)
    response = generate(prompt)
    
    try:
        match = re.search(r'\[.*\]', response, re.DOTALL)
        if match:
            return json.loads(match.group())
    except json.JSONDecodeError:
        pass
    
    print(f"Warning: Could not parse flashcard JSON")
    return []

In [ ]:
# Generate flashcards
all_flashcards = []

for i, chunk in enumerate(chunks):
    print(f"Generating flashcards for chunk {i+1}/{len(chunks)}...")
    cards = generate_flashcards(chunk, n=8)
    all_flashcards.extend(cards)
    print(f"  → {len(cards)} flashcards generated")

print(f"\nTotal: {len(all_flashcards)} flashcards")

# Preview
for i, card in enumerate(all_flashcards[:3]):
    print(f"\nCard {i+1}:")
    print(f"  Front: {card.get('front', 'N/A')}")
    print(f"  Back:  {card.get('back', 'N/A')}")

## 6. Memory Game Pairs

Generate matching pairs for the memory card game.

In [ ]:
MEMORY_PROMPT = """You are a Georgian language education assistant.
Based on the following text, generate exactly {n} matching pairs for a memory card game.

TEXT:
{text}

Return ONLY valid JSON array. Each item must have:
- "term": a key concept (short, 1-3 words, Georgian)
- "match": its matching definition or example (short, 1-5 words, Georgian)

Example:
[
  {{"term": "int", "match": "მთელი რიცხვი"}},
  {{"term": "for", "match": "ციკლი"}}
]

Generate {n} pairs. Return ONLY the JSON array."""


def generate_memory_pairs(text: str, n: int = 8) -> list[dict]:
    """Generate memory game pairs from text."""
    prompt = MEMORY_PROMPT.format(text=text[:3000], n=n)
    response = generate(prompt)
    
    try:
        match = re.search(r'\[.*\]', response, re.DOTALL)
        if match:
            return json.loads(match.group())
    except json.JSONDecodeError:
        pass
    
    print(f"Warning: Could not parse memory game JSON")
    return []

In [ ]:
# Generate memory pairs
all_pairs = []

for i, chunk in enumerate(chunks):
    print(f"Generating memory pairs for chunk {i+1}/{len(chunks)}...")
    pairs = generate_memory_pairs(chunk, n=8)
    all_pairs.extend(pairs)
    print(f"  → {len(pairs)} pairs generated")

print(f"\nTotal: {len(all_pairs)} memory pairs")

for i, p in enumerate(all_pairs[:5]):
    print(f"  {p.get('term', '?')} ↔ {p.get('match', '?')}")

## 7. Export Tutor Data

Bundle all generated content into a single JSON file — this powers the offline PWA.

In [ ]:
tutor_data = {
    "meta": {
        "title": "GeoTutor - Personal AI Tutor",
        "source": os.path.basename(PDF_PATH) if os.path.exists(PDF_PATH) else "sample_text",
        "language": "ka",
        "generated_by": "Gemma 3 4B via GeoTutor",
        "total_questions": len(all_questions),
        "total_flashcards": len(all_flashcards),
        "total_memory_pairs": len(all_pairs),
    },
    "quiz": all_questions,
    "flashcards": all_flashcards,
    "memory_pairs": all_pairs,
}

output_path = "/kaggle/working/tutor_data.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(tutor_data, f, ensure_ascii=False, indent=2)

size_kb = os.path.getsize(output_path) / 1024
print(f"Saved to {output_path} ({size_kb:.1f} KB)")
print(f"  Quiz:      {len(all_questions)} questions")
print(f"  Flashcards: {len(all_flashcards)} cards")
print(f"  Memory:    {len(all_pairs)} pairs")

## 8. Offline PWA Generator

Generate a complete offline web app (HTML + JS + CSS) that works without internet.
The tutor data is embedded directly into the HTML file.

In [ ]:
def generate_pwa(tutor_data: dict, output_dir: str = "/kaggle/working/geotutor-pwa"):
    """Generate a complete offline PWA from tutor data."""
    os.makedirs(output_dir, exist_ok=True)
    
    data_json = json.dumps(tutor_data, ensure_ascii=False)
    
    html = f"""<!DOCTYPE html>
<html lang="ka">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>GeoTutor</title>
<link rel="manifest" href="manifest.json">
<meta name="theme-color" content="#4F46E5">
<style>
* {{ margin: 0; padding: 0; box-sizing: border-box; }}
body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; background: #0f172a; color: #e2e8f0; min-height: 100vh; }}

.header {{ background: linear-gradient(135deg, #4F46E5, #7C3AED); padding: 20px; text-align: center; }}
.header h1 {{ font-size: 24px; margin-bottom: 4px; }}
.header p {{ font-size: 14px; opacity: 0.8; }}

.tabs {{ display: flex; background: #1e293b; border-bottom: 2px solid #334155; }}
.tab {{ flex: 1; padding: 14px 8px; text-align: center; cursor: pointer; font-size: 14px; transition: all 0.2s; border-bottom: 3px solid transparent; }}
.tab:hover {{ background: #334155; }}
.tab.active {{ border-bottom-color: #818CF8; color: #818CF8; background: #1e293b; }}

.content {{ max-width: 700px; margin: 0 auto; padding: 20px; }}
.screen {{ display: none; }}
.screen.active {{ display: block; }}

/* Quiz */
.quiz-card {{ background: #1e293b; border-radius: 16px; padding: 24px; margin-bottom: 16px; }}
.quiz-card h3 {{ color: #c7d2fe; margin-bottom: 16px; font-size: 18px; line-height: 1.5; }}
.quiz-progress {{ font-size: 13px; color: #94a3b8; margin-bottom: 8px; }}
.option {{ display: block; width: 100%; padding: 14px 16px; margin: 8px 0; background: #334155; border: 2px solid #475569; border-radius: 12px; color: #e2e8f0; font-size: 16px; cursor: pointer; text-align: left; transition: all 0.2s; }}
.option:hover {{ border-color: #818CF8; background: #3b4252; }}
.option.correct {{ background: #065f46; border-color: #10b981; }}
.option.wrong {{ background: #7f1d1d; border-color: #ef4444; }}
.explanation {{ background: #1a1a2e; border-left: 4px solid #818CF8; padding: 12px 16px; margin-top: 16px; border-radius: 0 8px 8px 0; font-size: 14px; line-height: 1.6; }}
.btn {{ display: inline-block; padding: 12px 32px; background: #4F46E5; color: white; border: none; border-radius: 12px; font-size: 16px; cursor: pointer; margin-top: 16px; }}
.btn:hover {{ background: #4338CA; }}

/* Flashcards */
.flashcard-container {{ perspective: 1000px; margin: 20px 0; }}
.flashcard {{ width: 100%; min-height: 200px; position: relative; cursor: pointer; transition: transform 0.6s; transform-style: preserve-3d; }}
.flashcard.flipped {{ transform: rotateY(180deg); }}
.flashcard .front, .flashcard .back {{ position: absolute; width: 100%; min-height: 200px; backface-visibility: hidden; border-radius: 16px; display: flex; align-items: center; justify-content: center; padding: 32px; font-size: 20px; line-height: 1.5; text-align: center; }}
.flashcard .front {{ background: linear-gradient(135deg, #4F46E5, #7C3AED); }}
.flashcard .back {{ background: linear-gradient(135deg, #059669, #10b981); transform: rotateY(180deg); }}
.flash-nav {{ display: flex; justify-content: space-between; align-items: center; margin-top: 16px; }}
.flash-counter {{ font-size: 14px; color: #94a3b8; }}
.category-tag {{ display: inline-block; background: #334155; padding: 4px 12px; border-radius: 20px; font-size: 12px; color: #818CF8; margin-bottom: 12px; }}

/* Memory Game */
.memory-grid {{ display: grid; grid-template-columns: repeat(4, 1fr); gap: 10px; margin: 16px 0; }}
.memory-card {{ aspect-ratio: 1; background: #334155; border-radius: 12px; display: flex; align-items: center; justify-content: center; cursor: pointer; font-size: 14px; padding: 8px; text-align: center; transition: all 0.3s; border: 2px solid #475569; word-break: break-word; }}
.memory-card.revealed {{ background: #4F46E5; border-color: #818CF8; }}
.memory-card.matched {{ background: #065f46; border-color: #10b981; cursor: default; }}
.memory-stats {{ display: flex; justify-content: space-around; margin: 16px 0; font-size: 14px; color: #94a3b8; }}

/* Progress */
.stat-card {{ background: #1e293b; border-radius: 16px; padding: 20px; margin-bottom: 12px; }}
.stat-card h4 {{ color: #818CF8; margin-bottom: 8px; }}
.progress-bar {{ width: 100%; height: 8px; background: #334155; border-radius: 4px; overflow: hidden; margin: 8px 0; }}
.progress-fill {{ height: 100%; background: linear-gradient(90deg, #4F46E5, #10b981); border-radius: 4px; transition: width 0.5s; }}
.score-big {{ font-size: 48px; font-weight: bold; text-align: center; margin: 20px 0; }}
</style>
</head>
<body>

<div class="header">
  <h1>GeoTutor</h1>
  <p>შენი პერსონალური AI მასწავლებელი</p>
</div>

<div class="tabs">
  <div class="tab active" onclick="showScreen('quiz')">Quiz</div>
  <div class="tab" onclick="showScreen('flashcards')">Flashcards</div>
  <div class="tab" onclick="showScreen('memory')">Memory</div>
  <div class="tab" onclick="showScreen('progress')">Progress</div>
</div>

<div class="content">
  <!-- Quiz Screen -->
  <div id="quiz" class="screen active">
    <div class="quiz-progress" id="quiz-progress"></div>
    <div class="quiz-card" id="quiz-card"></div>
  </div>

  <!-- Flashcards Screen -->
  <div id="flashcards" class="screen">
    <div id="flash-category" class="category-tag"></div>
    <div class="flashcard-container">
      <div class="flashcard" id="flashcard" onclick="flipCard()">
        <div class="front" id="flash-front"></div>
        <div class="back" id="flash-back"></div>
      </div>
    </div>
    <p style="text-align:center;font-size:13px;color:#64748b;margin-top:12px;">დააკლიკე კარტს გადასაბრუნებლად</p>
    <div class="flash-nav">
      <button class="btn" onclick="prevCard()">← წინა</button>
      <span class="flash-counter" id="flash-counter"></span>
      <button class="btn" onclick="nextCard()">შემდეგი →</button>
    </div>
  </div>

  <!-- Memory Game Screen -->
  <div id="memory" class="screen">
    <div class="memory-stats">
      <span>სვლები: <strong id="mem-moves">0</strong></span>
      <span>წყვილები: <strong id="mem-pairs">0</strong>/<strong id="mem-total">0</strong></span>
    </div>
    <div class="memory-grid" id="memory-grid"></div>
    <button class="btn" onclick="initMemory()" style="width:100%;">თავიდან დაწყება</button>
  </div>

  <!-- Progress Screen -->
  <div id="progress" class="screen">
    <div class="score-big" id="total-score">0%</div>
    <div id="progress-stats"></div>
  </div>
</div>

<script>
const DATA = {data_json};

// --- State ---
let quizIndex = 0, quizScore = 0, quizAnswered = false;
let flashIndex = 0;
let memCards = [], memFirst = null, memMoves = 0, memMatched = 0, memLocked = false;

// --- Navigation ---
function showScreen(id) {{
  document.querySelectorAll('.screen').forEach(s => s.classList.remove('active'));
  document.querySelectorAll('.tab').forEach(t => t.classList.remove('active'));
  document.getElementById(id).classList.add('active');
  event.target.classList.add('active');
}}

// --- Quiz ---
function renderQuiz() {{
  const q = DATA.quiz[quizIndex];
  if (!q) {{ document.getElementById('quiz-card').innerHTML = `<h3>Quiz დასრულდა!</h3><p>შედეგი: ${{quizScore}}/${{DATA.quiz.length}}</p><button class="btn" onclick="quizIndex=0;quizScore=0;renderQuiz();">თავიდან</button>`; updateProgress(); return; }}
  quizAnswered = false;
  document.getElementById('quiz-progress').textContent = `კითხვა ${{quizIndex+1}} / ${{DATA.quiz.length}}  |  სწორი: ${{quizScore}}`;
  let html = `<h3>${{q.question}}</h3>`;
  q.options.forEach((opt, i) => {{
    html += `<button class="option" onclick="checkAnswer(${{i}}, ${{q.correct}}, this)">${{String.fromCharCode(65+i)}}) ${{opt}}</button>`;
  }});
  html += `<div class="explanation" id="explanation" style="display:none"></div>`;
  html += `<button class="btn" id="next-btn" style="display:none" onclick="quizIndex++;renderQuiz();">შემდეგი →</button>`;
  document.getElementById('quiz-card').innerHTML = html;
}}

function checkAnswer(selected, correct, el) {{
  if (quizAnswered) return;
  quizAnswered = true;
  const q = DATA.quiz[quizIndex];
  const btns = el.parentElement.querySelectorAll('.option');
  btns[correct].classList.add('correct');
  if (selected === correct) {{ quizScore++; }}
  else {{ el.classList.add('wrong'); }}
  const exp = document.getElementById('explanation');
  exp.textContent = q.explanation || '';
  exp.style.display = 'block';
  document.getElementById('next-btn').style.display = 'inline-block';
  saveProgress();
}}

// --- Flashcards ---
function renderFlashcard() {{
  const card = DATA.flashcards[flashIndex];
  if (!card) return;
  document.getElementById('flashcard').classList.remove('flipped');
  document.getElementById('flash-front').textContent = card.front;
  document.getElementById('flash-back').textContent = card.back;
  document.getElementById('flash-counter').textContent = `${{flashIndex+1}} / ${{DATA.flashcards.length}}`;
  document.getElementById('flash-category').textContent = card.category || '';
}}

function flipCard() {{ document.getElementById('flashcard').classList.toggle('flipped'); }}
function nextCard() {{ flashIndex = (flashIndex + 1) % DATA.flashcards.length; renderFlashcard(); }}
function prevCard() {{ flashIndex = (flashIndex - 1 + DATA.flashcards.length) % DATA.flashcards.length; renderFlashcard(); }}

// --- Memory Game ---
function initMemory() {{
  memMoves = 0; memMatched = 0; memFirst = null; memLocked = false;
  const pairs = DATA.memory_pairs.slice(0, 6);
  memCards = [];
  pairs.forEach((p, i) => {{
    memCards.push({{ id: i*2, pairId: i, text: p.term, revealed: false, matched: false }});
    memCards.push({{ id: i*2+1, pairId: i, text: p.match, revealed: false, matched: false }});
  }});
  // Shuffle
  for (let i = memCards.length - 1; i > 0; i--) {{
    const j = Math.floor(Math.random() * (i + 1));
    [memCards[i], memCards[j]] = [memCards[j], memCards[i]];
  }}
  document.getElementById('mem-total').textContent = pairs.length;
  renderMemory();
}}

function renderMemory() {{
  document.getElementById('mem-moves').textContent = memMoves;
  document.getElementById('mem-pairs').textContent = memMatched;
  const grid = document.getElementById('memory-grid');
  grid.innerHTML = '';
  memCards.forEach((c, i) => {{
    const div = document.createElement('div');
    div.className = 'memory-card' + (c.revealed ? ' revealed' : '') + (c.matched ? ' matched' : '');
    div.textContent = (c.revealed || c.matched) ? c.text : '?';
    div.onclick = () => memClick(i);
    grid.appendChild(div);
  }});
}}

function memClick(i) {{
  if (memLocked || memCards[i].revealed || memCards[i].matched) return;
  memCards[i].revealed = true;
  renderMemory();
  if (memFirst === null) {{ memFirst = i; return; }}
  memMoves++;
  if (memCards[memFirst].pairId === memCards[i].pairId) {{
    memCards[memFirst].matched = true;
    memCards[i].matched = true;
    memMatched++;
    memFirst = null;
    renderMemory();
    if (memMatched === Math.floor(memCards.length / 2)) saveProgress();
  }} else {{
    memLocked = true;
    setTimeout(() => {{
      memCards[memFirst].revealed = false;
      memCards[i].revealed = false;
      memFirst = null;
      memLocked = false;
      renderMemory();
    }}, 800);
  }}
}}

// --- Progress ---
function saveProgress() {{
  const p = JSON.parse(localStorage.getItem('geotutor_progress') || '{{}}');
  p.quizScore = quizScore;
  p.quizTotal = DATA.quiz.length;
  p.memoryBest = Math.min(p.memoryBest || 999, memMoves);
  p.sessions = (p.sessions || 0) + 1;
  localStorage.setItem('geotutor_progress', JSON.stringify(p));
}}

function updateProgress() {{
  const p = JSON.parse(localStorage.getItem('geotutor_progress') || '{{}}');
  const pct = p.quizTotal ? Math.round((p.quizScore / p.quizTotal) * 100) : 0;
  document.getElementById('total-score').textContent = pct + '%';
  document.getElementById('progress-stats').innerHTML = `
    <div class="stat-card"><h4>Quiz შედეგი</h4>
      <div class="progress-bar"><div class="progress-fill" style="width:${{pct}}%"></div></div>
      <p>${{p.quizScore || 0}} / ${{p.quizTotal || 0}} სწორი</p></div>
    <div class="stat-card"><h4>Memory Game</h4>
      <p>საუკეთესო: ${{p.memoryBest === 999 ? '-' : p.memoryBest}} სვლა</p></div>
    <div class="stat-card"><h4>სესიები</h4>
      <p>სულ: ${{p.sessions || 0}}</p></div>
  `;
}}

// --- Init ---
renderQuiz();
if (DATA.flashcards.length) renderFlashcard();
if (DATA.memory_pairs.length) initMemory();
updateProgress();

// Register Service Worker
if ('serviceWorker' in navigator) {{
  navigator.serviceWorker.register('sw.js').catch(() => {{}});
}}
</script>
</body>
</html>"""
    
    # Service Worker for offline
    sw_js = """const CACHE = 'geotutor-v1';
const ASSETS = ['/', '/index.html'];
self.addEventListener('install', e => e.waitUntil(caches.open(CACHE).then(c => c.addAll(ASSETS))));
self.addEventListener('fetch', e => e.respondWith(caches.match(e.request).then(r => r || fetch(e.request))));"""
    
    # Manifest
    manifest = json.dumps({
        "name": "GeoTutor",
        "short_name": "GeoTutor",
        "start_url": "/",
        "display": "standalone",
        "background_color": "#0f172a",
        "theme_color": "#4F46E5",
        "icons": []
    }, indent=2)
    
    with open(f"{output_dir}/index.html", "w", encoding="utf-8") as f:
        f.write(html)
    with open(f"{output_dir}/sw.js", "w") as f:
        f.write(sw_js)
    with open(f"{output_dir}/manifest.json", "w") as f:
        f.write(manifest)
    
    print(f"PWA generated in {output_dir}/")
    print(f"  index.html ({len(html)/1024:.1f} KB)")
    print(f"  sw.js (service worker)")
    print(f"  manifest.json")
    return output_dir

pwa_dir = generate_pwa(tutor_data)
print(f"\nDownload the PWA folder and open index.html — works fully offline!")

## 9. Interactive Preview

Preview the generated content right here in the notebook.

In [ ]:
# Preview Quiz
print("=" * 60)
print("QUIZ PREVIEW")
print("=" * 60)
for i, q in enumerate(all_questions):
    print(f"\n{'─'*40}")
    print(f"Q{i+1}: {q['question']}")
    for j, opt in enumerate(q['options']):
        marker = '✅' if j == q['correct'] else '  '
        print(f"  {marker} {chr(65+j)}) {opt}")
    if q.get('explanation'):
        print(f"  💡 {q['explanation']}")

print(f"\n\n{'='*60}")
print("FLASHCARDS PREVIEW")
print("=" * 60)
for i, card in enumerate(all_flashcards):
    print(f"\n[{card.get('category', '')}]")
    print(f"  Front: {card['front']}")
    print(f"  Back:  {card['back']}")

print(f"\n\n{'='*60}")
print("MEMORY PAIRS PREVIEW")
print("=" * 60)
for p in all_pairs:
    print(f"  {p['term']}  ↔  {p['match']}")

In [ ]:
# Show PWA preview as iframe in notebook
with open(f"{pwa_dir}/index.html", "r", encoding="utf-8") as f:
    html_content = f.read()

display(HTML(f"""
<div style="border:2px solid #4F46E5; border-radius:16px; overflow:hidden; max-width:400px; margin:20px auto;">
  <iframe srcdoc='{html_content.replace(chr(39), "&#39;")}' 
          width="400" height="700" style="border:none;"></iframe>
</div>
<p style="text-align:center;color:#666;">PWA Preview (interactive!)</p>
"""))

## 10. Summary & Impact

### What GeoTutor does:
1. **PDF Upload** → Extracts Georgian text
2. **Gemma 3 AI** → Generates Quiz, Flashcards, Memory Game in Georgian
3. **Offline PWA** → Works without internet on any device

### Impact Metrics:
- **Digital Equity:** First Georgian-language AI tutor from any PDF
- **Offline Access:** PWA works in rural areas without internet
- **Scalable:** Any PDF in any language → personalized tutor

### Technical Stack:
| Component | Technology |
|-----------|------------|
| AI Model | Gemma 3 4B (4-bit quantized) |
| PDF Parser | PyPDF2 |
| Compute | Kaggle T4 GPU |
| Output | PWA (HTML/CSS/JS) |
| Offline | Service Worker + localStorage |

### Potential Reach:
- 5,000+ university students in Tbilisi
- 3,000+ rural school children
- 1,000+ course creators
- **9,000+ Georgians with AI-powered education**